# 05 - Results Comparison

This is the notebook that answers the project's **Question** from the SCQA in `00_scqa_overview.ipynb` directly: how much better is the layered approach than a naive EOQ baseline, at the same service level?

**Status: structure only** — fill in once Layers A/B (and ideally C) have run end-to-end. Left incomplete deliberately rather than filled with placeholder numbers that could be mistaken for real results.

In [ ]:
import sys
sys.path.append('..')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

model_inputs = pd.read_csv('../data/processed/model_inputs.csv')
policies = pd.read_csv('../data/processed/layer_a_policies.csv')
pooling = pd.read_csv('../data/processed/layer_b_pooling_results.csv')

## Baseline: static EOQ + fixed reorder point

$$EOQ = \sqrt{\frac{2 K \bar{D}}{h}}, \qquad ROP = \bar{D} \cdot L$$

(No safety stock term in the naive baseline — this is intentionally the 'ignore uncertainty' comparison point.)

In [ ]:
def eoq_baseline(row):
    h = row['unit_cost_placeholder'] * row['holding_cost_rate_annual'] / 365.0
    eoq = np.sqrt(2 * row['order_cost_fixed'] * row['demand_mean'] / h)
    rop = row['demand_mean'] * row['lead_time_mean_days']
    return pd.Series({'eoq': eoq, 'rop': rop})

baseline = model_inputs.apply(eoq_baseline, axis=1)
baseline.describe()

## TODO: total cost simulation

For a fair comparison, simulate each policy (EOQ/ROP, Layer A (s,S), Layer A+B pooled) over the same held-out demand trace (e.g. last 6 months of `train.csv`, not used to fit demand_mean/std) and record:
- realized service level (% of demand met from stock)
- total holding cost
- total stockout cost
- total ordering cost

This is the honest way to make the comparison — comparing *policy parameters* (like s vs ROP) directly is not an apples-to-apples cost comparison.

In [ ]:
# Simulation harness - TODO
# def simulate_policy(demand_trace, policy_fn, lead_time, cost_params): ...
# results = {
#     'EOQ baseline': simulate_policy(...),
#     'Layer A (s,S)': simulate_policy(...),
#     'Layer A+B pooled': simulate_policy(...),
# }
pass

## Final chart (target)

Bar chart: total cost per policy, at matched service level ~95%. This chart is the single artifact that should go into the app's 'Cost Trade-off' tab (`app/app.py`, tab4).